In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

_, base = expfolder( "system" );

In [ ]:
# Hyper parameters.
N = 128;  ρ = 0.01;  s = 0.01

# Extract critical noise.
folder = findfolder( N, ρ, s; base=base, crit=true )
η = readdlm( folder*"../order-0.5.txt" )[1];
folder

In [ ]:
# Import area and perimeter scaling values.
β = vec( readdlm( folder*"area-perimeter-scaling.txt" ) )[1:2]

# Unpack the probability map data.
ℓ = polyloglaw.( 1:N; β=β )
A = (ℓ.^2)./(4π);

In [ ]:
# Import the cluster change frequency values.
θlist = readdlm( folder*"optimal-rates-hat.txt" );

# Extract the power law.
λ = readdlm( folder*"power-law.txt" )[1]

# Construct the probability map.
Ω = [P( j; N=N, i=i, λℓ=θlist[i,1], λr=θlist[i,2], κ=θlist[i,3] ) for i in 1:N, j in 1:N];

In [ ]:
# Import cluster mass distribution extracted from data.
F = round.( defInt, vec( readdlm( folder*"cluster-frequency.txt" ) ) )
f = F./sum( F )

# Solve for the expected cluster masses at steady state.
n = N.*f./(1:N);

In [ ]:
# Create kernels from the calibration parameters.
function createkernels(N::defInt, A::Vector{defFloat}, ℓ::Vector{defFloat}, γ::Vector{defFloat}; f0::defFloat=100.0)::Tuple{Function,Function}
    # Construct coagulation kernel functions.
    coll(i::defInt, j::defInt)::defFloat = collision( ℓ, i, j; c0=γ[1], γ=1.0 )

    # Construct fragmentation kernel functions.
    frag(i::defInt)::defFloat = fragment( A, i; f0=f0, γ=0.0 )#, γ=θ[4] )

    # Return function variables.
    return coll, frag
end

# Master equation for cluster collision and fragmentation dynamics.
function model!(ṅ, n, params::Params, t::defFloat)
    # Reset the derivatives.
    fill!( ṅ, 0.0 )

    # For each potential cluster size.
    @threads for i ∈ 1:params.N
        # Cluster j > i fragments into clusters of size i and j-i.
        ṅ[i] += i < params.N ? sum( params.frag( j )*params.Ω[j,i]*n[j] for j ∈ i+1:params.N )          : 0.0

        # Cluster of size i fragments into clusters j and i-j.
        ṅ[i] -= i > 1        ? params.frag( i )/2*sum( params.Ω[i,j] for j ∈ 1:i-1 )*n[i]               : 0.0

        # Cluster of size j < i collides with cluster i-j to create i.
        ṅ[i] += i > 1        ? sum( params.coll( j,i-j )*j/i*params.Ω[j,i]*n[j]*n[i-j] for j ∈ 1:i-1 )  : 0.0

        # Cluster of size i collides with cluster j ≤ N-i to create i+j.
        ṅ[i] -= i < params.N ? sum( params.coll( i,j )*params.Ω[i,i+j]*n[i]*n[j] for j ∈ 1:params.N-i ) : 0.0
    end
end

# Simulate the model and return the cluster mass distribution.
function simulate(params::Params;
    f::Vector{defFloat}=fill( 1/params.N, params.N ),
    T::Real=10.0, verbose::Bool=false)

    # Initial conditions (from the given initial distribution).
    n0 = params.N/sum( i*f[i] for i ∈ 1:params.N )*f

    # Build the ODE problem and solve.
    problem = ODEProblem( model!, n0, T, params )
    results = solve( problem, Vern9(); isoutofdomain=(n,p,t) -> any( n .< 0 ),
        save_everystep=false, save_start=false, save_end=true )

    # Unpack the final state.
    n = results.u[end]

    # Return the final step of the simulation.
    Δn = clustersizedistr( n, params )
    return verbose ? (results, Δn) : Δn
end

function kernelloss(F::Vector{defInt}, γ::Vector{defFloat}, data; T::defFloat=10.0, verbose::Bool=false)
    # Unpack data variables.
    N, Ω, A, ℓ = data

    # Define local kernels.
    coll, frag = createkernels( N, A, ℓ, γ )
    params = Params( N, Ω, coll, frag )

    # Simulate the master equations.
    f = simulate( params; f=F./sum( F ), T=T )

    # Return the negative log-likelihood.
    NLL = nll( f, F )
    return verbose ? (NLL, f) : NLL
end

In [ ]:
# Compute the cluster mass distribution.
T = 10.0;  γ0 = [2.545]#, 0.91]
results = optimize( createkernelloss( F, N, Ω, A, ℓ; T=T ), log10.( γ0 ); show_trace=false )

# Unpack solution.
γ = 10.0.^results.minimizer

In [ ]:
A

In [ ]:
sum( .!isfinite.( A ) )

In [ ]:
# Unpack parameters and simulate for plotting.
NLL, f̂ = kernelloss( F, γ, (N, Ω, A, ℓ); T=5T, verbose=true )

In [ ]:
# Plot bounds.
xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( f[f .> 0] ) ) )

# Plot the final distribution from the mean-field equations.
plt = plot(; size=(400,250) )

plot!( plt, [N, N], 10.0.^[ymin,0]; color=:black, lw=2, label="" )

# Plot the CMD from simulations and master equations.
scatter!( plt, f; color=:gray, label="raw data" )
scatter!( plt, f̂; color=:black, lw=2, label="master eq." )

plot!( plt; xlims=10.0.^[0,xmax], xscale=:log10 )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)", legend=:bottomleft )

# saveplot( plt, "../figures/cluster-distribution-model_N-$(N).png" )

In [ ]:
# Save the kernel exponents and coefficients.
if false
    writedlm( folder*"kernel-parameters-$(length( γ )).txt", γ )
end